In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
'''for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))'''

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

"for dirname, _, filenames in os.walk('/kaggle/input'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))"

# Imports and data preprocessing

In [2]:
import os 
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Normalization and Transformation

In [4]:
from torchvision.transforms import InterpolationMode
train_transform = transforms.Compose([
    transforms.Resize([224], interpolation=InterpolationMode.BICUBIC),  
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.RandomVerticalFlip(p=0.5),  
    transforms.RandomRotation(degrees=30), 
    transforms.RandomCrop([224], padding=4),
    transforms.ColorJitter(
    brightness=0.5, 
    contrast=0.5,   
    saturation=0.5,  
    hue=0.1
),
    transforms.ToTensor(),  
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])
val_transform = transforms.Compose([
    transforms.Resize([224], interpolation=InterpolationMode.BICUBIC),  
    transforms.CenterCrop([224]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

In [5]:
im_dir = "/kaggle/input/dlp-ga-9-image-classification/train"
train_dataset = datasets.ImageFolder(im_dir,transform=train_transform)

In [6]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)

# Model building

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

model = models.vit_h_14(
    weights = models.ViT_H_14_Weights.IMAGENET1K_SWAG_LINEAR_V1
)

Downloading: "https://download.pytorch.org/models/vit_h_14_lc_swag-c1eb923e.pth" to /root/.cache/torch/hub/checkpoints/vit_h_14_lc_swag-c1eb923e.pth
100%|██████████| 2.35G/2.35G [00:28<00:00, 90.1MB/s]


In [8]:
for param in model.parameters():
    param.requires_grad = False
model.heads = nn.Sequential(
    nn.Linear(
        in_features=1280,
        out_features = 128,
        bias=True
    ),
    nn.BatchNorm1d(128),
    nn.GELU(),
    nn.Dropout(0.25),
    nn.Linear(
        in_features=128,
        out_features=38,
        bias=True
    )
)
for param in model.heads.parameters():
    param.requires_grad = True

for param in model.encoder.layers.encoder_layer_31.parameters():
    param.requires_grad = True

In [9]:
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model) 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Using 2 GPUs!


In [10]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=0.01)

In [11]:
import torch
from sklearn.metrics import f1_score
from tqdm import tqdm

def evaluate_model(model, dataloader, device):
    model.eval() 
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validating Model", total=len(dataloader)):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    correct = (all_preds == all_labels).sum().item()
    total = all_labels.size(0)
    accuracy = correct / total
    f1 = f1_score(all_labels.numpy(), all_preds.numpy(), average="macro")
    return accuracy, f1

# Model training 

In [17]:
from tqdm import tqdm
import torch
from torch.cuda.amp import GradScaler, autocast
import os

num_epochs = 3
scaler = GradScaler()
torch.cuda.empty_cache()
for epoch in range(num_epochs):
    model.train()  
    train_loss = 0
    with tqdm(train_dataloader, desc=f"Epoch [{epoch+1}/{num_epochs}]", unit="batch") as pbar:
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            pbar.set_postfix({"loss": f"{train_loss/len(train_dataloader):.4f}"})

/tmp/ipykernel_48/781599737.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch [1/3]:   0%|          | 0/679 [00:00<?, ?batch/s]/tmp/ipykernel_48/781599737.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch [3/3]: 100%|██████████| 679/679 [18:57<00:00,  1.68s/batch, loss=0.1227]


# Final Submission

In [14]:
from torchvision.io import read_image
import pandas as pd
from PIL import Image
from tqdm import tqdm
import os
import torch

def classify_images_to_csv(image_folder, model, transform, output_csv):
    model.eval()
    results = []
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    total = len(image_files)
    for image_name in tqdm(image_files, desc="Processing Images", total=total):
        image_path = os.path.join(image_folder, image_name)
        image = Image.open(image_path).convert("RGB")
        image = transform(image).unsqueeze(0)
        with torch.no_grad():
            outputs = model(image) 
            probabilities = torch.softmax(outputs, dim=1) 
            label = torch.argmax(probabilities, dim=1).item()  
        results.append({
            "Image_ID": image_name.split('.')[0],
            "Label": label
        })
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"Predictions saved to {output_csv}")

In [18]:
classify_images_to_csv('/kaggle/input/dlp-ga-9-image-classification/test',model,val_transform,"submission2.csv")

Processing Images: 100%|██████████| 10876/10876 [34:42<00:00,  5.22it/s]


Predictions saved to submission2.csv
